# 05 — Ensemble STEMI trên PTB-XL + ACS-ECG (kiểu WBF)

**Dự án ACS-ECG-AI (Vinmec)** · **Bài toán 2** — phát hiện nhồi máu cơ tim có ST chênh lên (STEMI)

Notebook này **gộp hai nguồn dữ liệu khác nhau** — *ACS-ECG Dataset 2026* (Du et al., Sci Data
13:1009, 2026) và **PTB-XL** (tải qua Kaggle) — huấn luyện **9 kiến trúc 1D CNN/Transformer**
trên cùng một tập gộp, rồi hợp nhất xác suất bằng **ensemble kiểu WBF** (mỗi model là một nguồn
xác suất, trọng số theo AUROC validation của chính nó — lấy cảm hứng từ Weighted Boxes Fusion
trong object detection, áp cho bài toán phân loại nhị phân).

> ⚠️ **PTB-XL không có nhãn STEMI thật.** Notebook dùng siêu lớp chẩn đoán `MI` của PTB-XL làm
> **nhãn proxy** — xem cảnh báo chi tiết ở mục 5. Đây là domain/label shift có chủ đích, không
> phải nhãn chuẩn vàng; bảng đánh giá tách riêng theo nguồn ở mục 17 tồn tại chính để đo ảnh hưởng
> này.

> Notebook tự chứa, chạy được trên **Google Colab** lẫn **máy cá nhân**. Không sửa 01–04 —
> pipeline đa nguồn dữ liệu này tách hẳn để không phá vỡ 4 notebook đang chạy tốt kia.

## 1. Cài thư viện

`wfdb` để đọc tín hiệu WFDB (cả ACS-ECG lẫn PTB-XL đều dùng định dạng này), `kagglehub` để tải
PTB-XL từ Kaggle. Trên Colab cần cài lại mỗi phiên (~10-20 giây).

In [ ]:
import importlib.util
import subprocess
import sys

for _pkg in ("wfdb", "kagglehub"):
    if importlib.util.find_spec(_pkg) is None:
        print(f"Đang cài {_pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

import kagglehub
import wfdb

print("wfdb", wfdb.__version__, "| kagglehub", kagglehub.__version__)

## 2. Cấu hình

Notebook chạy được trên **Google Colab** và **máy cá nhân**, tự nhận diện môi trường.
Chỉ cần sửa đúng một chỗ:

- **Colab** → `DRIVE_PROJECT` (tên thư mục trên Drive chứa `datasets.zip` của ACS-ECG)
- **Máy cá nhân** → `LOCAL_DATA_ROOT`

Ngoài ra đổi `RUN_MODE` khi muốn chuyển từ chạy thử sang chạy thật (xem mục cuối cùng).

**PTB-XL không đi qua Drive** — dataset nhỏ hơn ACS-ECG nhiều (~1,7 GB, ít file lớn chứ không phải
hàng chục nghìn file nhỏ), `kagglehub` tải thẳng và tự cache, không cần bước zip thủ công.

**Thư mục `outputs/` chỉ giữ hai thứ:** checkpoint mô hình và cache tín hiệu. Hình và bảng số
đều hiển thị ngay trong notebook.

In [ ]:
import os
from pathlib import Path

import torch

# ===================== CHẾ ĐỘ CHẠY =====================
RUN_MODE = "full"        # "debug" = ~400 bản ghi / 3 epoch  |  "full" = toàn bộ / 30 epoch

# ===================== ĐƯỜNG DẪN =======================
IS_COLAB = importlib.util.find_spec("google.colab") is not None

if IS_COLAB:
    DRIVE_PROJECT = Path("/content/drive/MyDrive/ACS-ECG-AI")   # <<< SỬA nếu đặt tên khác
    DRIVE_DATA_ZIP = DRIVE_PROJECT / "datasets.zip"             # <<< SỬA nếu tên zip khác
    DATA_ROOT = Path("/content/datasets")      # ACS-ECG giải nén ra đĩa local cho nhanh
    WORK_DIR = Path("/content/work")           # tạm, mất khi hết phiên
    PERSIST_DIR = DRIVE_PROJECT / "outputs"    # bền qua các phiên
else:
    LOCAL_DATA_ROOT = r"C:\Users\anhquan\Workspace\AI_THUC_CHIEN\VSF_Projects\ECG-experiment\datasets"                    # <<< SỬA khi chạy máy cá nhân
    DATA_ROOT = Path(LOCAL_DATA_ROOT)
    WORK_DIR = DATA_ROOT.parent / "outputs"
    PERSIST_DIR = WORK_DIR

CACHE_DIR = WORK_DIR / "cache"                             # cache tín hiệu (sinh lại được)
MODEL_DIR = PERSIST_DIR / "models" / "stemi_ensemble_ptbxl"  # checkpoint (không được mất)
DRIVE_CACHE_DIR = (PERSIST_DIR / "cache") if IS_COLAB else None

# ===================== THAM SỐ =========================
SEED = 42
FS, SIGNAL_LEN, NUM_LEADS = 500, 5000, 12      # 12 chuyển đạo × 10 giây @ 500 Hz — khớp cả 2 nguồn
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 40.0, 3       # bandpass Butterworth
VAL_SIZE = 0.15
LR, WEIGHT_DECAY = 1e-3, 1e-4
EARLY_STOP_PATIENCE, LR_PATIENCE = 10, 5
TARGET_LABEL = "label"
CLASS_NAME = "STEMI/MI-proxy"

KAGGLE_DATASET = "m0hamedyousry/ptb-xl-a-large-publicly-available-ecg-dataset"

# Sanity check riêng cho phần ACS-ECG (paper tính trên toàn bộ 19.955, ta chỉ có 90% có nhãn)
PAPER_STEMI_COUNT, SANITY_TOL = 1513, 0.10

# ===================== SUY RA ==========================
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if GPU_AVAILABLE else "cpu")
DEBUG_LIMIT = 400        # cao hơn 01/03 (200) vì cần đủ mẫu cho 4 tổ hợp (nguồn × nhãn)
EPOCHS = 3 if RUN_MODE == "debug" else 30
BATCH_SIZE = 8 if RUN_MODE == "debug" else (64 if GPU_AVAILABLE else 16)
USE_AMP = GPU_AVAILABLE

print("Môi trường :", "Google Colab" if IS_COLAB else "máy cá nhân")
print("Thiết bị   :", DEVICE)
print("Chế độ     :", RUN_MODE, f"({EPOCHS} epoch, batch {BATCH_SIZE})")
print("Nhãn đích  :", CLASS_NAME, "(STEMI ở ACS-ECG, MI-proxy ở PTB-XL — xem mục 5)")

### 2b. Chuẩn bị dữ liệu ACS-ECG

Giống hệt cách 01/03 làm: trên máy cá nhân chỉ tạo thư mục; trên Colab mount Drive (popup xin
quyền — thao tác thủ công duy nhất cho phần này), copy `datasets.zip` về `/content` rồi giải nén.

Phải đi vòng qua zip vì bộ dữ liệu có **59.867 file**, mà Drive tính mỗi lần mở file là một lệnh
gọi mạng — đọc trực tiếp từ Drive sẽ chậm hơn hàng chục lần.

In [ ]:
import shutil
import time
import zipfile


def _locate_data_root(search_root: Path) -> Path:
    """Tìm thư mục thật chứa dữ liệu ACS-ECG, chịu được việc zip có bọc thêm lớp thư mục."""
    wanted = {"csv", "row_data", "raw_data", "med_data"}
    best, best_score = search_root, -1
    for cand in [search_root, *[p for p in search_root.rglob("*") if p.is_dir()]]:
        try:
            names = {c.name.lower() for c in cand.iterdir() if c.is_dir()}
        except OSError:
            continue
        if len(names & wanted) > best_score:
            best, best_score = cand, len(names & wanted)
        if best_score >= 2:
            break
    return best


def stage_colab_data() -> Path:
    marker = DATA_ROOT / ".staged_ok"
    if marker.exists():
        real = Path(marker.read_text().strip())
        print("Dữ liệu ACS-ECG đã sẵn sàng:", real)
        return real

    if not DRIVE_DATA_ZIP.exists():
        có_gì = sorted(p.name for p in DRIVE_PROJECT.iterdir())[:40]
        raise FileNotFoundError(
            f"Không thấy {DRIVE_DATA_ZIP}\nTrong {DRIVE_PROJECT} hiện có: {có_gì}"
        )

    local_zip = Path("/content/_dataset.zip")
    size = DRIVE_DATA_ZIP.stat().st_size
    if not (local_zip.exists() and local_zip.stat().st_size == size):
        print(f"Copy zip {size / 1024 ** 3:.2f} GB từ Drive ...")
        t0 = time.time()
        shutil.copy2(DRIVE_DATA_ZIP, local_zip)
        print(f"  {time.time() - t0:.0f}s")

    extract_to = Path("/content/_extract")
    if extract_to.exists():
        shutil.rmtree(extract_to)
    print("Giải nén ...")
    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(extract_to)
    print(f"  {time.time() - t0:.0f}s")

    real = _locate_data_root(extract_to)
    real.mkdir(parents=True, exist_ok=True)
    (real / ".staged_ok").write_text(str(real))
    return real


if IS_COLAB:
    if str(DRIVE_PROJECT).startswith(("http://", "https://", "www.")):
        raise ValueError(
            "DRIVE_PROJECT đang là URL chia sẻ Drive. Phải là đường dẫn sau khi mount:\n"
            '    DRIVE_PROJECT = Path("/content/drive/MyDrive/ACS-ECG-AI")'
        )
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    if not DRIVE_PROJECT.exists():
        my = Path("/content/drive/MyDrive")
        có_gì = sorted(p.name for p in my.iterdir() if p.is_dir())[:40] if my.exists() else []
        raise FileNotFoundError(
            f"Không thấy {DRIVE_PROJECT}\nCác thư mục trong MyDrive: {có_gì}"
        )

for d in [CACHE_DIR, MODEL_DIR] + ([DRIVE_CACHE_DIR] if DRIVE_CACHE_DIR else []):
    d.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    DATA_ROOT = stage_colab_data()

print("ACS-ECG DATA_ROOT:", DATA_ROOT, "| tồn tại:", DATA_ROOT.exists())

### 2c. Chuẩn bị dữ liệu PTB-XL (Kaggle)

`kagglehub.dataset_download()` cần credential Kaggle — bước thủ công một lần, tương tự popup mount
Drive ở trên. Có 3 cách, notebook tự dò theo thứ tự:

1. **Colab (khuyến nghị):** mở tab hình chìa khoá bên trái > *Secrets* > thêm `KAGGLE_USERNAME`
   và `KAGGLE_KEY` (lấy tại `kaggle.com/settings` > API > *Create New Token*, mở file
   `kaggle.json` vừa tải để lấy 2 giá trị này).
2. **Mọi môi trường:** tải `kaggle.json` từ `kaggle.com/settings` > API, đặt vào
   `~/.kaggle/kaggle.json` (máy cá nhân nhớ `chmod 600`).
3. Set biến môi trường `KAGGLE_USERNAME` và `KAGGLE_KEY` trước khi chạy notebook.

Thiếu cả ba thì cell dưới dừng lại với `RuntimeError` in rõ hướng dẫn — không đoán mò.

In [ ]:
def _find_kaggle_credentials():
    """Trả về mô tả nguồn credential đã tìm thấy, hoặc None nếu không có."""
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return "biến môi trường KAGGLE_USERNAME/KAGGLE_KEY"

    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if kaggle_json.exists():
        return f"{kaggle_json}"

    if IS_COLAB:
        try:
            from google.colab import userdata
            username, key = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
            if username and key:
                os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = username, key
                return "Colab Secrets (KAGGLE_USERNAME/KAGGLE_KEY)"
        except Exception:
            pass
    return None


_cred_src = _find_kaggle_credentials()
if _cred_src is None:
    raise RuntimeError(
        "Không tìm thấy Kaggle credentials — cần MỘT trong ba cách sau:\n"
        "  1) Colab: tab hình chìa khoá bên trái > Secrets > thêm KAGGLE_USERNAME và KAGGLE_KEY\n"
        "     (lấy tại kaggle.com/settings > API > Create New Token).\n"
        "  2) Mọi môi trường: tải kaggle.json từ kaggle.com/settings > API, đặt vào\n"
        "     ~/.kaggle/kaggle.json (máy cá nhân nhớ chmod 600).\n"
        "  3) Set biến môi trường KAGGLE_USERNAME và KAGGLE_KEY trước khi chạy notebook."
    )
print(f"Kaggle credentials: OK ({_cred_src})")

_ptbxl_cache = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("PTB-XL tải/lấy cache tại:", _ptbxl_cache)


def _locate_ptbxl_root(search_root: Path) -> Path:
    for cand in [search_root, *[p for p in search_root.rglob("*") if p.is_dir()]]:
        if (cand / "ptbxl_database.csv").exists():
            return cand
    raise FileNotFoundError(f"Không thấy ptbxl_database.csv trong {search_root}")


PTBXL_DIR = _locate_ptbxl_root(_ptbxl_cache)
assert (PTBXL_DIR / "records500").exists(), f"Không thấy records500/ trong {PTBXL_DIR}"
print("PTB-XL data root:", PTBXL_DIR)

## 3. Seed & GPU

In [ ]:
import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Input cố định (12 × 5000) nên cuDNN chọn thuật toán nhanh nhất một lần rồi giữ nguyên.
torch.backends.cudnn.benchmark = GPU_AVAILABLE

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if GPU_AVAILABLE else "không có")
if IS_COLAB and not GPU_AVAILABLE:
    print("\n" + "!" * 70)
    print("!! COLAB KHÔNG CÓ GPU -> Runtime > Change runtime type > GPU, rồi Run all lại.")
    print("!" * 70)

## 4. Nhãn ACS-ECG

Giống 01/03: nhãn `STEMI` lấy trực tiếp từ `train.csv` (phần 90% chính thức có nhãn).
`test.csv` là tập test ẩn, **không bao giờ được đọc** trong notebook này.

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)


def find_dir(root: Path, names):
    wanted = {n.lower() for n in names}
    for c in sorted(root.iterdir()):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    for c in root.rglob("*"):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    return None


RAW_DIR = find_dir(DATA_ROOT, ["row_data", "raw_data"])
CSV_DIR = find_dir(DATA_ROOT, ["CSV", "csv"])
assert RAW_DIR and CSV_DIR, f"Không thấy row_data/ hoặc CSV/ trong {DATA_ROOT}"

acs_raw = pd.read_csv(CSV_DIR / "train.csv")
acs_raw["record_stem"] = acs_raw["ecg_row_record"].astype(str).str.replace(".dat", "", regex=False)

COL_PATIENT, COL_AGE, COL_GENDER = "Patient_id", "age", "gender"

acs_df = pd.DataFrame({
    "source": "acs_ecg",
    "record_id": acs_raw["record_stem"],
    "record_path": acs_raw["record_stem"].apply(lambda s: str(RAW_DIR / s)),
    "patient_key": "acs_ecg:" + acs_raw[COL_PATIENT].astype(str),
    "age": acs_raw[COL_AGE],
    "gender_m": acs_raw[COL_GENDER].astype(int),
    "label": acs_raw["STEMI"].astype(int),
})

print(f"ACS-ECG: {len(acs_df):,} bản ghi | {acs_raw[COL_PATIENT].nunique():,} bệnh nhân")
_got = int(acs_df["label"].sum())
_delta = (_got - PAPER_STEMI_COUNT) / PAPER_STEMI_COUNT
print(f"  STEMI dương: {_got:,} ({acs_df['label'].mean():.2%}) | paper: {PAPER_STEMI_COUNT:,} "
      f"| lệch {_delta:+.1%} {'OK' if abs(_delta) <= SANITY_TOL else '!! LỆCH QUÁ 10%'}")

## 5. Nhãn PTB-XL (proxy) & hoà trộn 2 nguồn

**⚠️ PTB-XL không có nhãn STEMI trực tiếp.** Notebook dùng siêu lớp chẩn đoán `MI` (suy từ
`scp_codes` qua `scp_statements.csv`, đúng theo cách làm mẫu chính thức của PTB-XL — không lọc
theo likelihood) làm **nhãn proxy `MI_PROXY`**.

Nhãn này **không tương đương** STEMI cấp cứu định nghĩa theo DSA của ACS-ECG: nó gồm cả nhồi máu
cũ, không phân biệt có/không ST chênh. Trộn 2 định nghĩa vào một nhãn nhị phân là **domain/label
shift có chủ đích** — chấp nhận vì mục tiêu của notebook là khảo sát ensemble đa nguồn, không phải
tạo ra một bộ nhãn chuẩn vàng mới. Bảng đánh giá tách riêng theo `source` ở mục 17 tồn tại chính
để đo ảnh hưởng của việc này — **đừng đọc AUROC gộp một mình mà bỏ qua bảng đó.**

PTB-XL mã hoá giới tính ngược với ACS-ECG (`sex`: 0=nam, 1=nữ; ACS-ECG `gender`: 1=nam, 0=nữ) —
notebook đảo lại (`1 - sex`) để cột `gender_m` nhất quán giữa 2 nguồn.

In [ ]:
import ast

ptbxl_db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col="ecg_id")
ptbxl_db["scp_codes"] = ptbxl_db["scp_codes"].apply(ast.literal_eval)

scp_stmt = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)
agg_df = scp_stmt[scp_stmt["diagnostic"] == 1]


def aggregate_diagnostic(scp_dict: dict) -> set:
    """Suy ra siêu lớp chẩn đoán (NORM/MI/STTC/CD/HYP) từ các mã SCP-ECG của bản ghi."""
    classes = set()
    for code_ in scp_dict:
        if code_ in agg_df.index:
            classes.add(agg_df.loc[code_, "diagnostic_class"])
    return classes


ptbxl_db["diagnostic_classes"] = ptbxl_db["scp_codes"].apply(aggregate_diagnostic)
ptbxl_db["MI_PROXY"] = ptbxl_db["diagnostic_classes"].apply(lambda s: int("MI" in s))

ptbxl_df = pd.DataFrame({
    "source": "ptbxl",
    "record_id": ptbxl_db.index.astype(str),
    "record_path": ptbxl_db["filename_hr"].apply(lambda p: str(PTBXL_DIR / p)),
    "patient_key": "ptbxl:" + ptbxl_db["patient_id"].astype(int).astype(str),
    "age": ptbxl_db["age"],
    "gender_m": 1 - ptbxl_db["sex"].astype(int),
    "label": ptbxl_db["MI_PROXY"].astype(int),
})

print(f"PTB-XL: {len(ptbxl_df):,} bản ghi | {ptbxl_db['patient_id'].nunique():,} bệnh nhân")
print(f"  MI_PROXY dương: {ptbxl_df['label'].sum():,} ({ptbxl_df['label'].mean():.2%})")

df_all = pd.concat([acs_df, ptbxl_df], ignore_index=True)
df_all["strata"] = df_all["source"] + "_" + df_all["label"].astype(str)

print(f"\nGỘP: {len(df_all):,} bản ghi | {df_all['patient_key'].nunique():,} bệnh nhân")
display(df_all.groupby(["source", "label"]).size().rename("n").reset_index())

## 6. Lấy mẫu khi chạy thử

Ở `RUN_MODE="debug"` lấy `DEBUG_LIMIT` bản ghi **stratified theo `(nguồn, nhãn)`** — 4 tổ hợp —
để tập mẫu không rơi vào toàn 0/toàn 1 ở bất kỳ nguồn nào (khi đó metric của nguồn đó vô định).

In [ ]:
from sklearn.model_selection import train_test_split

if RUN_MODE == "debug" and len(df_all) > DEBUG_LIMIT:
    df, _ = train_test_split(df_all, train_size=DEBUG_LIMIT,
                             stratify=df_all["strata"], random_state=SEED)
    df = df.reset_index(drop=True)
else:
    df = df_all.reset_index(drop=True)

print(f"Dùng {len(df):,} bản ghi")
display(df.groupby(["source", "label"]).size().rename("n").reset_index())
assert df["label"].nunique() == 2, "Tập chỉ có một lớp — không train được."
assert df["source"].nunique() == 2, "Tập chỉ có một nguồn — kiểm tra lại debug sampling."

## 7. Nhìn nhanh dữ liệu

Biểu đồ cuối là lý do phải chia train/val **theo bệnh nhân** (mục 9); `patient_key` đã gộp cả 2
nguồn nên không lẫn bệnh nhân dù trùng số ID gốc.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))

ct = df.groupby(["source", "label"]).size().unstack(fill_value=0)
ct.plot(kind="bar", ax=ax[0], color=["#4C78A8", "#E45756"])
ax[0].set_title("Phân phối nhãn theo nguồn")
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=0)
ax[0].legend(["âm (0)", "dương (1)"])

age_plot = df.copy()
age_plot["age"] = age_plot["age"].clip(upper=95)   # PTB-XL mã hoá >89 tuổi thành số lớn bất thường
for src, c in [("acs_ecg", "#4C78A8"), ("ptbxl", "#E45756")]:
    ax[1].hist(age_plot.loc[age_plot["source"] == src, "age"], bins=25, alpha=.6, label=src, color=c)
ax[1].set_title("Tuổi theo nguồn")
ax[1].set_xlabel("tuổi")
ax[1].legend()

rp = df.groupby("patient_key").size().value_counts().sort_index()
ax[2].bar(rp.index.astype(str), rp.values, color="#72B7B2")
ax[2].set_title("Số ECG mỗi bệnh nhân (2 nguồn gộp)")
ax[2].set_yscale("log")

plt.tight_layout()
plt.show()

## 8. Tiền xử lý và cache

Với mỗi bản ghi: đọc WFDB → NaN→0 → cắt/đệm về đúng 5000 mẫu → bandpass Butterworth 0,5–40 Hz
(`filtfilt`, zero-phase) — giống hệt 01/03.

**Hai cache riêng biệt** (`CACHE_ACS`, `CACHE_PTBXL`) vì 2 nguồn có cấu trúc thư mục khác nhau.
Cache ACS-ECG dùng lại đúng tiền tố `sig_` và cách hash theo `record_id` như 01/03 — nếu bạn đã
chạy một trong hai notebook đó ở `RUN_MODE="full"`, cache được dùng lại thẳng, không tốn công tiền
xử lý ACS-ECG lần nữa.

> Hai bản ghi lỗi đã biết ở ACS-ECG (`03228`, `14262`, xem 01/03) được xử lý bằng cùng cơ chế bắt
> `ValueError` rồi zero-pad phần thiếu.

In [ ]:
import hashlib
import json

from scipy.signal import butter, filtfilt

_B, _A = butter(BP_ORDER, [BP_LOW / (FS / 2), BP_HIGH / (FS / 2)], btype="band")
TRUNCATED = []


def load_signal(path: str) -> np.ndarray:
    try:
        rec = wfdb.rdrecord(path)
    except ValueError:                     # .dat ngắn hơn header khai báo
        n_sig = int(Path(path + ".hea").read_text().splitlines()[0].split()[1])
        n = Path(path + ".dat").stat().st_size // (n_sig * 2)
        rec = wfdb.rdrecord(path, sampto=n)
        TRUNCATED.append(path)
    return np.asarray(rec.p_signal, dtype=np.float32).T


def preprocess(sig: np.ndarray) -> np.ndarray:
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    if sig.shape[1] < SIGNAL_LEN:
        sig = np.pad(sig, ((0, 0), (0, SIGNAL_LEN - sig.shape[1])))
    return np.ascontiguousarray(filtfilt(_B, _A, sig[:, :SIGNAL_LEN], axis=1), dtype=np.float32)


def _complete(meta_p: Path, npy_p: Path, n_expected: int) -> bool:
    if not (meta_p.exists() and npy_p.exists()):
        return False
    try:
        return json.loads(meta_p.read_text()).get("n_done", 0) >= n_expected
    except (OSError, ValueError):
        return False


def build_cache(prefix: str, ids: list, paths: list) -> np.ndarray:
    """Cache tiền xử lý riêng theo nguồn. `prefix="sig"` (ACS-ECG) khớp đúng quy ước 01/03
    nên dùng lại được cache đã build ở notebook kia nếu record_id + thứ tự trùng khớp."""
    _hash = hashlib.md5("|".join(ids).encode()).hexdigest()[:8]
    npy_p = CACHE_DIR / f"{prefix}_{RUN_MODE}_n{len(ids)}_{_hash}.npy"
    meta_p = CACHE_DIR / f"{prefix}_{RUN_MODE}_n{len(ids)}_{_hash}.meta.json"

    if DRIVE_CACHE_DIR and not _complete(meta_p, npy_p, len(ids)):
        d_npy, d_meta = DRIVE_CACHE_DIR / npy_p.name, DRIVE_CACHE_DIR / meta_p.name
        if _complete(d_meta, d_npy, len(ids)):
            print(f"[{prefix}] lấy cache từ Drive ({d_npy.stat().st_size / 1024 ** 3:.2f} GB) ...")
            shutil.copy2(d_npy, npy_p)
            shutil.copy2(d_meta, meta_p)

    if _complete(meta_p, npy_p, len(ids)):
        print(f"[{prefix}] cache đã đầy đủ:", npy_p.name)
        return np.load(npy_p, mmap_mode="r")

    meta = json.loads(meta_p.read_text()) if meta_p.exists() else {}
    start = int(meta.get("n_done", 0)) if npy_p.exists() else 0
    if start:
        arr = np.lib.format.open_memmap(npy_p, mode="r+")
        print(f"[{prefix}] build tiếp từ {start}/{len(ids)}")
    else:
        print(f"[{prefix}] build cache {len(ids):,} bản ghi "
              f"(~{len(ids) * NUM_LEADS * SIGNAL_LEN * 2 / 1024 ** 3:.2f} GB)")
        arr = np.lib.format.open_memmap(npy_p, mode="w+", dtype=np.float16,
                                        shape=(len(ids), NUM_LEADS, SIGNAL_LEN))

    t0 = time.time()
    step = max(1, len(ids) // 8)
    for i in range(start, len(ids)):
        arr[i] = preprocess(load_signal(paths[i])).astype(np.float16)
        if (i + 1) % step == 0 or i + 1 == len(ids):
            arr.flush()
            meta_p.write_text(json.dumps({"n_done": i + 1}))
            el = max(time.time() - t0, 1e-6)
            done = i + 1 - start
            print(f"  [{prefix}] {i + 1:>6,}/{len(ids):,}  {done / el:5.0f} rec/s  "
                  f"ETA {(len(ids) - i - 1) / (done / el):4.0f}s")
    del arr
    print(f"[{prefix}] xong trong {time.time() - t0:.0f}s")

    if DRIVE_CACHE_DIR:
        print(f"[{prefix}] sao lưu cache lên Drive ...")
        shutil.copy2(npy_p, DRIVE_CACHE_DIR / npy_p.name)
        shutil.copy2(meta_p, DRIVE_CACHE_DIR / meta_p.name)
    return np.load(npy_p, mmap_mode="r")


df_acs = df[df["source"] == "acs_ecg"].reset_index(drop=True)
df_ptbxl = df[df["source"] == "ptbxl"].reset_index(drop=True)

CACHE_ACS = build_cache("sig", df_acs["record_id"].tolist(), df_acs["record_path"].tolist())
CACHE_PTBXL = build_cache("ptbxl", df_ptbxl["record_id"].tolist(), df_ptbxl["record_path"].tolist())

# ghép lại + gắn (nguồn cache, chỉ số cục bộ trong nguồn đó) cho từng dòng
df = pd.concat([df_acs, df_ptbxl], ignore_index=True)
df["cache_idx"] = list(range(len(df_acs))) + list(range(len(df_ptbxl)))


def get_signal(row_idx: int) -> np.ndarray:
    row = df.loc[row_idx]
    cache = CACHE_ACS if row["source"] == "acs_ecg" else CACHE_PTBXL
    return np.asarray(cache[row["cache_idx"]], dtype=np.float32)


print(f"\nCACHE_ACS {CACHE_ACS.shape}  |  CACHE_PTBXL {CACHE_PTBXL.shape}")
if TRUNCATED:
    print(f"Bản ghi bị cắt ngắn, đã zero-pad: {len(TRUNCATED)}")

## 9. Chia train/val theo bệnh nhân

Patient-level split trên `patient_key` (đã gộp `nguồn:id` nên không lẫn bệnh nhân giữa 2 nguồn dù
trùng số ID gốc). Stratify theo `strata` = `(nguồn, nhãn)` ở **cấp bệnh nhân** để giữ tỷ lệ 2
nguồn cân đối ở cả train lẫn val.

z-score fit trên **toàn bộ train gộp** (không tách riêng theo nguồn) — đơn giản, nhất quán với
01/03; domain gap giữa 2 nguồn được nhìn qua bảng breakdown ở mục 17, không phải ở bước chuẩn hoá.

In [ ]:
pat = df.groupby("patient_key")["label"].max().reset_index()
pat_strata = df.drop_duplicates("patient_key").set_index("patient_key")["strata"]
pat["strata"] = pat["patient_key"].map(pat_strata)
strat = pat["strata"] if pat["strata"].value_counts().min() >= 2 else None
pat_tr, pat_va = train_test_split(pat, test_size=VAL_SIZE, stratify=strat, random_state=SEED)

train_idx = df.index[df["patient_key"].isin(set(pat_tr["patient_key"]))].to_numpy()
val_idx = df.index[df["patient_key"].isin(set(pat_va["patient_key"]))].to_numpy()
y = df["label"].values.astype(np.float32)

assert not (set(pat_tr["patient_key"]) & set(pat_va["patient_key"])), "RÒ RỈ: bệnh nhân ở cả 2 tập"
print("Giao nhau patient_key: RỖNG -> OK\n")

display(pd.DataFrame({
    "tập": ["train", "val"],
    "bệnh nhân": [len(pat_tr), len(pat_va)],
    "bản ghi": [len(train_idx), len(val_idx)],
    "dương": [int(y[train_idx].sum()), int(y[val_idx].sum())],
    "tỷ lệ dương": [f"{y[train_idx].mean():.2%}", f"{y[val_idx].mean():.2%}"],
}))
display(df.loc[val_idx].groupby("source").size().rename("bản ghi val theo nguồn").reset_index())


def _fetch_batch(idx_arr):
    return np.stack([get_signal(i) for i in idx_arr])


def norm_stats(idx, chunk=256):
    order = np.sort(np.asarray(idx))
    s = np.zeros(NUM_LEADS)
    ss = np.zeros(NUM_LEADS)
    cnt = 0
    for i in range(0, len(order), chunk):
        b = _fetch_batch(order[i:i + chunk]).astype(np.float64)
        s += b.sum(axis=(0, 2))
        ss += (b ** 2).sum(axis=(0, 2))
        cnt += b.shape[0] * b.shape[2]
    m = s / cnt
    return m.astype(np.float32), np.sqrt(np.maximum(ss / cnt - m ** 2, 1e-12)).astype(np.float32)


LEAD_MEAN, LEAD_STD = norm_stats(train_idx)
print(f"\nz-score fit trên {len(train_idx):,} bản ghi TRAIN (gộp 2 nguồn)")

## 10. Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader, Dataset


class ECGDataset(Dataset):
    def __init__(self, indices, labels):
        self.indices = np.asarray(indices)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.mean = LEAD_MEAN.reshape(-1, 1)
        self.std = LEAD_STD.reshape(-1, 1)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x = get_signal(self.indices[i])
        return torch.from_numpy((x - self.mean) / self.std), torch.tensor(self.labels[i])


# Windows tạo worker bằng spawn (mỗi worker nạp lại cả notebook) nên để 0; Linux dùng fork -> 2
NUM_WORKERS = 0 if os.name == "nt" else 2
_extra = dict(persistent_workers=True, prefetch_factor=4) if NUM_WORKERS else {}

train_loader = DataLoader(ECGDataset(train_idx, y[train_idx]), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)
val_loader = DataLoader(ECGDataset(val_idx, y[val_idx]), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)

xb, yb = next(iter(train_loader))
print(f"batch: {tuple(xb.shape)} {xb.dtype} | nhãn {tuple(yb.shape)} | "
      f"{len(train_loader)} batch train, {len(val_loader)} batch val")

## 11. Hàm tính metric

Giống hệt 01/03. `compute_metrics` an toàn với tập nhỏ (trả `nan` thay vì lỗi khi chỉ có một lớp).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)

plt.rcParams["figure.dpi"] = 110


def _div(a, b):
    return float(a) / float(b) if b else float("nan")


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=np.float64).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    two = len(np.unique(y_true)) == 2
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "auroc": roc_auc_score(y_true, y_prob) if two else float("nan"),
        "auprc": average_precision_score(y_true, y_prob) if two else float("nan"),
        "sensitivity": _div(tp, tp + fn), "specificity": _div(tn, tn + fp),
        "ppv": _div(tp, tp + fp), "npv": _div(tn, tn + fn),
        "f1": _div(2 * tp, 2 * tp + fp + fn),
        "accuracy": _div(tp + tn, tp + tn + fp + fn),
        "brier": float(brier_score_loss(y_true, y_prob)) if two else float("nan"),
        "threshold": float(threshold),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "n": int(len(y_true)), "n_pos": int(y_true.sum()),
    }


def per_class_report(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_pred = (np.asarray(y_prob).ravel() >= threshold).astype(int)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    rows = [
        {"Class": f"Negative (0) — không {CLASS_NAME}", "Precision": p[0], "Recall": r[0],
         "F1": f[0], "Support": int(s[0])},
        {"Class": f"Positive (1) — {CLASS_NAME}", "Precision": p[1], "Recall": r[1],
         "F1": f[1], "Support": int(s[1])},
    ]
    w = s / s.sum()
    rows.append({"Class": "macro avg", "Precision": p.mean(), "Recall": r.mean(),
                 "F1": f.mean(), "Support": int(s.sum())})
    rows.append({"Class": "weighted avg", "Precision": float(p @ w), "Recall": float(r @ w),
                 "F1": float(f @ w), "Support": int(s.sum())})
    return pd.DataFrame(rows).set_index("Class")


def style_df(df_, pct_cols):
    fmt = {c: "{:.4f}" for c in pct_cols}
    fmt.update({c: "{:,d}" for c in df_.columns if c not in pct_cols and df_[c].dtype.kind in "iu"})
    try:
        return df_.style.format(fmt, na_rep="n/a").background_gradient(
            cmap="Blues", vmin=0, vmax=1, subset=[c for c in pct_cols if c in df_.columns])
    except Exception:
        return df_.round(4)


print("self-test:", {k: round(v, 3) for k, v in
                     compute_metrics([0, 0, 1, 1], [.1, .4, .35, .8]).items()
                     if k in ("auroc", "f1", "sensitivity")})

## 12. Chín kiến trúc

Tất cả nhận input `(batch, 12, 5000)` và trả về **một logit** cho mỗi mẫu, dùng chung một vòng
lặp huấn luyện.

| mô hình | ý tưởng chính |
|---|---|
| **PlainCNN** | Mốc sàn: Conv-BN-ReLU-MaxPool xếp chồng, không có gì đặc biệt. |
| **ResNet1D** | Skip connection giúp gradient đi xuyên qua mạng sâu. |
| **InceptionTime1D** | Nhiều độ dài kernel song song (39/19/9) — bắt được cả sóng nhanh (QRS) lẫn biến thiên chậm (đoạn ST). |
| **CNN+BiLSTM** | CNN rút đặc trưng cục bộ, BiLSTM mô hình hoá quan hệ theo thời gian giữa các nhịp. |
| **SEResNet1D** | ResNet1D + squeeze-excitation sau mỗi khối — model tự học cách cân trọng các kênh đặc trưng. |
| **Transformer1D** | Cắt tín hiệu thành patch theo thời gian, self-attention — bắt phụ thuộc dài hạn mà CNN thuần khó nắm. |
| **TCN1D** | Conv1D giãn nở (dilation) nhiều tầng — receptive field rộng mà vẫn giữ độ phân giải thời gian cao. |
| **XResNet1D** | Biến thể ResNet1D: stem sâu (3 conv nhỏ xếp chồng), nhánh tắt AvgPool+1×1 conv, activation SiLU. |
| **ConvNeXtV2_1D** | Khối depthwise-conv + LayerNorm + MLP mở rộng + Global Response Normalization — bản 1D của ConvNeXt V2. |

In [ ]:
import torch.nn as nn


# ---------------------------------------------------------------- 1. PlainCNN
class PlainCNN(nn.Module):
    """Mốc sàn: Conv-BN-ReLU-MaxPool xếp chồng, không có gì đặc biệt."""

    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in channels:
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


# ---------------------------------------------------------------- 2. ResNet1D
class ResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)


class ResNet1D(nn.Module):
    """Skip connection giúp gradient đi xuyên qua mạng sâu."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(ResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 3. InceptionTime1D
class InceptionModule(nn.Module):
    """Nhiều độ dài kernel song song -> bắt được cả sóng nhanh lẫn biến thiên chậm."""

    def __init__(self, c_in, n_filters=32, kernels=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bottleneck = nn.Conv1d(c_in, bottleneck, 1, bias=False)
        self.convs = nn.ModuleList(
            [nn.Conv1d(bottleneck, n_filters, k, padding=k // 2, bias=False) for k in kernels])
        self.pool_conv = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1),
                                       nn.Conv1d(c_in, n_filters, 1, bias=False))
        self.bn = nn.BatchNorm1d(n_filters * (len(kernels) + 1))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        b = self.bottleneck(x)
        return self.relu(self.bn(torch.cat([c(b) for c in self.convs] + [self.pool_conv(x)], 1)))


class InceptionTime1D(nn.Module):
    def __init__(self, n_filters=32, depth=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 4, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True))
        c_out = n_filters * 4
        self.blocks = nn.ModuleList()
        self.shortcuts = nn.ModuleList()
        self.pools = nn.ModuleList()
        c_in = 32
        res_c = 32
        for d in range(depth):
            self.blocks.append(InceptionModule(c_in, n_filters))
            if d % 3 == 2:
                self.shortcuts.append(nn.Sequential(nn.Conv1d(res_c, c_out, 1, bias=False),
                                                    nn.BatchNorm1d(c_out)))
                self.pools.append(nn.MaxPool1d(4))
                res_c = c_out
            else:
                self.shortcuts.append(None)
                self.pools.append(None)
            c_in = c_out
        self.relu = nn.ReLU(inplace=True)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_out, 1))

    def forward(self, x):
        x = self.stem(x)
        res = x
        for blk, short, pool in zip(self.blocks, self.shortcuts, self.pools):
            x = blk(x)
            if short is not None:
                x = self.relu(x + short(res))
                x = pool(x)
                res = x
        return self.head(x).squeeze(-1)


# ---------------------------------------------------------------- 4. CNN + BiLSTM
class CNNBiLSTM(nn.Module):
    """CNN rút đặc trưng cục bộ, BiLSTM mô hình hoá quan hệ theo thời gian giữa các nhịp."""

    def __init__(self, hidden=128):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in (32, 64, 128):
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.cnn = nn.Sequential(*layers)
        self.lstm = nn.LSTM(c_in, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden * 2, 1))

    def forward(self, x):
        h = self.cnn(x).transpose(1, 2)
        out, _ = self.lstm(h)
        return self.head(out.mean(dim=1)).squeeze(-1)


# ---------------------------------------------------------------- 5. SEResNet1D
class SEBlock1D(nn.Module):
    """Squeeze-excitation: học trọng số quan trọng theo từng kênh (feature map)."""

    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels), nn.Sigmoid())

    def forward(self, x):
        w = self.fc(self.pool(x).squeeze(-1)).unsqueeze(-1)
        return x * w


class SEResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.se = SEBlock1D(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        out = self.se(self.bn2(self.conv2(out)))
        return self.relu(out + idt)


class SEResNet1D(nn.Module):
    """ResNet1D + squeeze-excitation sau mỗi khối."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(SEResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 6. Transformer1D
class Transformer1D(nn.Module):
    """Cắt tín hiệu thành patch theo thời gian, học vị trí, qua self-attention."""

    def __init__(self, patch_len=250, d_model=128, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        assert SIGNAL_LEN % patch_len == 0
        n_patches = SIGNAL_LEN // patch_len
        self.patch_len = patch_len
        self.proj = nn.Linear(NUM_LEADS * patch_len, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model * 4,
                                           dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x):
        b = x.shape[0]
        patches = x.unfold(-1, self.patch_len, self.patch_len)          # (B, 12, n_patches, patch_len)
        patches = patches.permute(0, 2, 1, 3).reshape(b, patches.shape[2], -1)
        tok = self.proj(patches)
        tok = torch.cat([self.cls_token.expand(b, -1, -1), tok], dim=1) + self.pos_embed
        out = self.norm(self.encoder(tok))
        return self.head(out[:, 0]).squeeze(-1)


# ---------------------------------------------------------------- 7. TCN1D
class TCNBlock1D(nn.Module):
    """Conv1D giãn nở (dilated), residual — receptive field rộng mà giữ độ phân giải thời gian."""

    def __init__(self, c_in, c_out, k=7, dilation=1, dropout=0.1):
        super().__init__()
        pad = (k - 1) * dilation // 2
        self.conv1 = nn.Conv1d(c_in, c_out, k, padding=pad, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, padding=pad, dilation=dilation, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = nn.Identity() if c_in == c_out else nn.Conv1d(c_in, c_out, 1, bias=False)

    def forward(self, x):
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.short(x))


class TCN1D(nn.Module):
    def __init__(self, channels=(32, 64, 128, 128, 256, 256), dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for i, c_out in enumerate(channels):
            blocks.append(TCNBlock1D(c_in, c_out, dilation=2 ** i, dropout=dropout))
            if i % 2 == 1:                       # hạ chiều thời gian định kỳ, không hạ mỗi lớp
                blocks.append(nn.MaxPool1d(2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 8. XResNet1D
class XResBlock1D(nn.Module):
    """Nhánh tắt kiểu xResNet: AvgPool + 1x1 conv thay vì strided conv khi downsample."""

    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.act = nn.SiLU(inplace=True)
        if stride == 1 and c_in == c_out:
            self.short = nn.Identity()
        else:
            self.short = nn.Sequential(
                nn.AvgPool1d(stride, ceil_mode=True), nn.Conv1d(c_in, c_out, 1, bias=False),
                nn.BatchNorm1d(c_out))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.act(self.bn1(self.conv1(x))))
        out = self.bn2(self.conv2(out))
        return self.act(out + idt)


class XResNet1D(nn.Module):
    """Stem sâu (3 conv nhỏ xếp chồng) thay vì 1 conv to + SiLU — biến thể tối ưu hoá của ResNet1D."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(NUM_LEADS, 32, 5, 2, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 32, 5, 1, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 32, 5, 1, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(XResBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.SiLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 9. ConvNeXtV2_1D
class GRN1D(nn.Module):
    """Global Response Normalization (ConvNeXt V2) — chuẩn hoá theo norm toàn cục của từng kênh."""

    def __init__(self, channels):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, channels))
        self.beta = nn.Parameter(torch.zeros(1, 1, channels))

    def forward(self, x):                          # x: (B, L, C)
        gx = torch.norm(x, p=2, dim=1, keepdim=True)
        nx = gx / (gx.mean(dim=-1, keepdim=True) + 1e-6)
        return self.gamma * (x * nx) + self.beta + x


class ConvNeXtV2Block1D(nn.Module):
    def __init__(self, channels, expand=4, dropout=0.1):
        super().__init__()
        self.dwconv = nn.Conv1d(channels, channels, 7, padding=3, groups=channels, bias=False)
        self.norm = nn.LayerNorm(channels)
        self.pw1 = nn.Linear(channels, channels * expand)
        self.act = nn.GELU()
        self.grn = GRN1D(channels * expand)
        self.pw2 = nn.Linear(channels * expand, channels)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):                           # x: (B, C, L)
        idt = x
        x = self.dwconv(x).transpose(1, 2)           # (B, L, C)
        x = self.norm(x)
        x = self.pw2(self.grn(self.act(self.pw1(x))))
        x = self.drop(x).transpose(1, 2)
        return x + idt


class ConvNeXtV2Downsample1D(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.norm = nn.BatchNorm1d(c_in)
        self.conv = nn.Conv1d(c_in, c_out, 2, stride=2, bias=False)

    def forward(self, x):
        return self.conv(self.norm(x))


class ConvNeXtV2_1D(nn.Module):
    """Khối depthwise-conv + LayerNorm + MLP mở rộng + GRN — bản 1D của ConvNeXt V2."""

    def __init__(self, channels=(32, 64, 128, 256), depths=(1, 1, 2, 1), dropout=0.1):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, channels[0], 4, stride=4, bias=False),
                                  nn.BatchNorm1d(channels[0]))
        stages, c_in = [], channels[0]
        for stage_i, (c_out, depth) in enumerate(zip(channels, depths)):
            if stage_i > 0:
                stages.append(ConvNeXtV2Downsample1D(c_in, c_out))
            stages += [ConvNeXtV2Block1D(c_out, dropout=dropout) for _ in range(depth)]
            c_in = c_out
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.LayerNorm(c_in), nn.Dropout(dropout), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.stages(self.stem(x))).squeeze(-1)


MODELS = {
    "PlainCNN": PlainCNN,
    "ResNet1D": ResNet1D,
    "InceptionTime1D": InceptionTime1D,
    "CNN+BiLSTM": CNNBiLSTM,
    "SEResNet1D": SEResNet1D,
    "Transformer1D": Transformer1D,
    "TCN1D": TCN1D,
    "XResNet1D": XResNet1D,
    "ConvNeXtV2_1D": ConvNeXtV2_1D,
}

rows = []
for name, cls in MODELS.items():
    m = cls().to(DEVICE)
    with torch.no_grad():
        out = m(torch.zeros(2, NUM_LEADS, SIGNAL_LEN, device=DEVICE))
    rows.append({"Mô hình": name, "Tham số": sum(p.numel() for p in m.parameters()),
                 "Output": str(tuple(out.shape))})
    del m
if GPU_AVAILABLE:
    torch.cuda.empty_cache()
display(pd.DataFrame(rows).set_index("Mô hình"))

## 13. Huấn luyện lần lượt chín mô hình

Mỗi mô hình dùng đúng một hàm, đúng một cấu hình: `BCEWithLogitsLoss` với `pos_weight` tính từ tỷ
lệ lớp trong train, AdamW, `ReduceLROnPlateau` theo val AUPRC, early stopping patience 10,
**gradient clipping (max-norm 1.0)**. Seed đặt lại trước mỗi model để khởi tạo trọng số không phụ
thuộc thứ tự chạy.

`Transformer1D` dùng LR thấp hơn riêng (`MODEL_LR_OVERRIDE`) vì self-attention hội tụ kém ổn định
hơn CNN/ResNet ở cùng LR — từng NaN giữa chừng khi dùng chung LR=1e-3 với các model còn lại.

**Chạy lại được sau khi mất phiên:** model nào đã có checkpoint khớp cấu hình thì được nạp lại và
bỏ qua train.

> ⚠️ **Cảnh báo thời gian chạy:** 9 model × ~35–40k bản ghi (2 nguồn gộp) lớn hơn đáng kể so với
> 03/04 (4 model × 18k). Full run trên Colab free T4 có thể mất **vài giờ**. Checkpoint-resume vẫn
> hoạt động — đứt phiên giữa chừng chỉ mất phần đang train dở, Run all lại sẽ tiếp tục đúng chỗ.

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

n_pos_tr = int(y[train_idx].sum())
POS_WEIGHT = float((len(train_idx) - n_pos_tr) / max(n_pos_tr, 1))
FINGERPRINT = dict(run_mode=RUN_MODE, target=TARGET_LABEL, n_train=len(train_idx), epochs=EPOCHS)
print(f"pos_weight = {POS_WEIGHT:.2f} | {EPOCHS} epoch tối đa mỗi mô hình\n")

# Self-attention hội tụ kém ổn định hơn CNN/ResNet ở cùng LR — Transformer1D từng NaN giữa chừng
# vì gradient nổ (loss 1,01 -> 2,14 chỉ sau 4 epoch rồi NaN). LR thấp hơn + grad clipping bên dưới
# là cách chuẩn để trị việc này, áp riêng cho model dùng attention thay vì hạ LR chung cho cả bộ.
MODEL_LR_OVERRIDE = {"Transformer1D": 3e-4}
GRAD_CLIP_NORM = 1.0


def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(xb)
            ys.append(yb.numpy())
            ps.append(torch.sigmoid(logits.float()).cpu().numpy())
    y_out, p_out = np.concatenate(ys), np.concatenate(ps)
    bad = ~np.isfinite(p_out)
    if bad.any():
        print(f"  !! {bad.sum()} xác suất không hữu hạn (NaN/Inf) -> thay bằng 0.5 để không crash "
              f"roc_auc_score; model này đang huấn luyện không ổn định.")
        p_out = np.where(bad, 0.5, p_out)
    return y_out, p_out


def train_one(name, cls):
    ckpt_path = MODEL_DIR / f"{name.replace('+', '_')}_best.pt"

    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        if ck.get("fp") == FINGERPRINT:
            model = cls().to(DEVICE)
            model.load_state_dict(ck["model"])
            print(f"[{name}] đã có checkpoint khớp -> bỏ qua train "
                  f"(best AUPRC {ck['best']:.4f} @ ep {ck['best_epoch']})")
            return model, ck["hist"], ck["best_epoch"], ck["seconds"]

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model = cls().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    lr = MODEL_LR_OVERRIDE.get(name, LR)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=.5, patience=LR_PATIENCE)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best, best_epoch, bad, hist = -np.inf, -1, 0, []
    t_all = time.time()
    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        tot, seen = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(xb), yb)
            if not torch.isfinite(loss):
                continue    # batch bệnh lý (hiếm) -> bỏ qua thay vì để làm hỏng toàn bộ trọng số
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            tot += loss.item() * xb.size(0)
            seen += xb.size(0)

        va_y, va_p = evaluate(model, val_loader)
        m = compute_metrics(va_y, va_p)
        scheduler.step(m["auprc"] if not np.isnan(m["auprc"]) else -np.inf)
        hist.append({"epoch": epoch + 1, "train_loss": tot / max(seen, 1),
                     "val_auprc": m["auprc"], "val_auroc": m["auroc"]})

        if (not np.isnan(m["auprc"])) and m["auprc"] > best:
            best, best_epoch, bad = float(m["auprc"]), epoch + 1, 0
            torch.save({"model": model.state_dict(), "fp": FINGERPRINT, "best": best,
                        "best_epoch": best_epoch, "hist": hist,
                        "seconds": round(time.time() - t_all, 1)}, ckpt_path)
        else:
            bad += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{name}] ep {epoch + 1:>3}/{EPOCHS}  {time.time() - t0:5.1f}s  "
                  f"loss {tot / max(seen, 1):.4f}  val auprc {m['auprc']:.4f}  auroc {m['auroc']:.4f}")
        if bad >= EARLY_STOP_PATIENCE:
            print(f"  [{name}] early stop @ epoch {epoch + 1}")
            break

    seconds = round(time.time() - t_all, 1)
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    ck["seconds"] = seconds
    ck["hist"] = hist
    torch.save(ck, ckpt_path)
    model.load_state_dict(ck["model"])
    print(f"[{name}] xong {len(hist)} epoch trong {seconds:.0f}s | "
          f"best AUPRC {best:.4f} @ ep {best_epoch}")
    return model, hist, best_epoch, seconds


RESULTS = {}
for name, cls in MODELS.items():
    print(f"\n=== {name} " + "=" * (60 - len(name)))
    model, hist, best_epoch, seconds = train_one(name, cls)
    va_y, va_p = evaluate(model, val_loader)
    RESULTS[name] = {"y": va_y, "p": va_p, "hist": hist,
                     "best_epoch": best_epoch, "seconds": seconds,
                     "params": sum(q.numel() for q in model.parameters())}
    del model
    if GPU_AVAILABLE:
        torch.cuda.empty_cache()

print("\nĐã train xong", len(RESULTS), "mô hình.")

## 14. Ensemble kiểu WBF

Mỗi model là một nguồn xác suất; trọng số = `max(AUROC_model - 0.5, eps)` (model tệ hơn ngẫu
nhiên gần như không đóng góp), chuẩn hoá tổng = 1, rồi lấy trung bình có trọng số — lấy cảm hứng
từ Weighted Boxes Fusion (hợp nhất nhiều dự đoán theo độ tin cậy), áp cho xác suất phân loại thay
vì bounding box.

So sánh thêm với trung bình đều (`Average-ensemble`) và model tốt nhất đơn lẻ, để trả lời câu hỏi
ensemble có thực sự lợi hay chỉ pha loãng model tốt nhất.

In [ ]:
_EPS = 1e-3
auroc_per_model = {name: compute_metrics(r["y"], r["p"])["auroc"] for name, r in RESULTS.items()}
w_raw = {name: max(a - 0.5, _EPS) for name, a in auroc_per_model.items()}
w_sum = sum(w_raw.values())
weights = {name: w / w_sum for name, w in w_raw.items()}

# Tất cả model đánh giá trên cùng val_loader (shuffle=False) nên nhãn phải khớp thứ tự tuyệt đối
val_y_ref = next(iter(RESULTS.values()))["y"]
assert all(np.array_equal(r["y"], val_y_ref) for r in RESULTS.values()), \
    "Nhãn validation lệch giữa các model — kiểm tra lại val_loader có bị shuffle không."

p_wbf = sum(weights[name] * RESULTS[name]["p"] for name in RESULTS)
p_avg = sum(RESULTS[name]["p"] for name in RESULTS) / len(RESULTS)
best_single = max(auroc_per_model, key=auroc_per_model.get)

RESULTS["WBF-ensemble"] = {"y": val_y_ref, "p": p_wbf}
RESULTS["Average-ensemble"] = {"y": val_y_ref, "p": p_avg}

print("Trọng số WBF (theo AUROC validation từng model, chuẩn hoá tổng = 1):")
for name, w in sorted(weights.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<16} AUROC={auroc_per_model[name]:.4f}  w={w:.4f}")
print(f"\nModel tốt nhất đơn lẻ: {best_single} (AUROC {auroc_per_model[best_single]:.4f})")

## 15. Bảng so sánh toàn bộ (9 model + 2 ensemble)

Cách đọc: **`AUROC`/`AUPRC` không phụ thuộc ngưỡng**, dùng chúng để xếp hạng. Ngưỡng mỗi hàng chọn
riêng theo Youden's J trên chính tập của hàng đó.

In [ ]:
val_source = df.loc[val_idx, "source"].to_numpy()

COLS = ["AUROC", "AUPRC", "F1", "Precision", "Sensitivity_Recall", "Specificity", "Accuracy"]


def _row(y_, p_):
    if len(np.unique(y_.astype(int))) == 2:
        fpr, tpr, thr = roc_curve(y_, p_)
        thr_best = float(np.clip(thr[int(np.argmax(tpr - fpr))], 0, 1))
    else:
        thr_best = 0.5
    m = compute_metrics(y_, p_, thr_best)
    return {"AUROC": m["auroc"], "AUPRC": m["auprc"], "F1": m["f1"], "Precision": m["ppv"],
            "Sensitivity_Recall": m["sensitivity"], "Specificity": m["specificity"],
            "Accuracy": m["accuracy"], "Threshold": thr_best, "N": m["n"], "N_pos": m["n_pos"]}


rows = {name: _row(r["y"], r["p"]) for name, r in RESULTS.items()}
compare_df = pd.DataFrame(rows).T[COLS + ["Threshold", "N", "N_pos"]].sort_values("AUPRC", ascending=False)
print("=" * 78)
print(f"SO SÁNH TOÀN BỘ MODEL + ENSEMBLE — tập validation {len(val_idx):,} bản ghi (2 nguồn gộp)")
print("=" * 78)
display(style_df(compare_df, COLS))

BEST_OVERALL = compare_df.index[0]
print(f"\n-> Tốt nhất theo AUPRC: {BEST_OVERALL} "
      f"(AUROC {compare_df.loc[BEST_OVERALL, 'AUROC']:.4f}, AUPRC {compare_df.loc[BEST_OVERALL, 'AUPRC']:.4f})")
if "WBF-ensemble" in compare_df.index and "Average-ensemble" in compare_df.index:
    wbf_auroc = compare_df.loc["WBF-ensemble", "AUROC"]
    best_single_auroc = auroc_per_model[best_single]
    print(f"WBF-ensemble AUROC {wbf_auroc:.4f} vs. model tốt nhất đơn lẻ ({best_single}) "
          f"AUROC {best_single_auroc:.4f}  |  chênh {wbf_auroc - best_single_auroc:+.4f}")

## 16. Breakdown theo nguồn dữ liệu

**Phần quan trọng nhất về mặt khoa học của notebook này.** AUROC gộp ở mục 15 có thể che khuất
việc model học tốt trên nguồn dễ (STEMI thật của ACS-ECG) nhưng kém trên nguồn khó/nhãn nhiễu hơn
(MI-proxy của PTB-XL) — hoặc ngược lại. Đừng kết luận gì về "STEMI" nói chung nếu chưa xem cả hai
bảng dưới.

In [ ]:
for src, ten in [("acs_ecg", "ACS-ECG (nhãn STEMI thật)"), ("ptbxl", "PTB-XL (nhãn MI-proxy)")]:
    mask = val_source == src
    rows_src = {name: _row(RESULTS[name]["y"][mask], RESULTS[name]["p"][mask]) for name in RESULTS}
    df_src = pd.DataFrame(rows_src).T[COLS + ["N", "N_pos"]].sort_values("AUPRC", ascending=False)
    print(f"\n--- {ten} — {mask.sum():,} bản ghi val, {int(y[val_idx][mask].sum()):,} dương ---")
    display(style_df(df_src, COLS))

## 17. Bảng tóm tắt gọn

Rút gọn từ bảng mục 15, chỉ giữ các metric chính — tiện để chụp màn hình hoặc dán vào báo cáo.

In [ ]:
summary_df = compare_df[["AUROC", "AUPRC", "F1", "Precision", "Sensitivity_Recall",
                         "Specificity", "Accuracy"]].rename(columns={"Sensitivity_Recall": "Sensitivity"})
summary_df.index.name = "Mô hình"
display(style_df(summary_df, summary_df.columns.tolist()))

## 18. Metric tách riêng từng lớp (model tốt nhất theo AUPRC)

Giống 01/03: tách lớp âm/dương để không bị đánh lừa bởi Accuracy cao trên bài toán mất cân bằng.

In [ ]:
_best = RESULTS[BEST_OVERALL]
_thr = compare_df.loc[BEST_OVERALL, "Threshold"]
print(f"{BEST_OVERALL}  |  ngưỡng {_thr:.4f}")
display(style_df(per_class_report(_best["y"], _best["p"], _thr), ["Precision", "Recall", "F1"]))

## 19. Lưu ý khi kết luận

**Nhãn proxy PTB-XL yếu hơn ACS-ECG** (mục 5) — mọi con số AUROC/AUPRC trên PTB-XL trong bảng
breakdown (mục 16) phản ánh khả năng phát hiện siêu lớp `MI` nói chung, **không phải STEMI cấp cứu
đúng nghĩa**. Không dùng những con số đó để khẳng định model "phát hiện STEMI tốt trên PTB-XL".

**Domain gap có thể khiến ensemble không thắng model tốt nhất đơn lẻ** — nếu 2 nguồn dữ liệu quá
khác biệt, hợp nhất xác suất có thể kéo lùi hiệu năng thay vì cải thiện. Đây là kết quả hợp lệ cần
báo cáo trung thực, không phải lý do chỉnh nhãn/trọng số WBF để "ép" kết quả đẹp hơn.

**Chỉ có một lần chia train/val**, chưa cross-validation, nên khoảng tin cậy của các metric chưa
được ước lượng. Seed cố định nhưng `cudnn.benchmark` + AMP khiến mỗi lần chạy lại lệch khoảng
±0,01–0,02 AUROC — hai model/ensemble cách nhau dưới mức đó nên coi là ngang nhau.

**Tập test ẩn 10% của ACS-ECG (`test.csv`) không được đụng tới** ở bất kỳ bước nào.

## 20. Chuyển từ chạy thử sang chạy thật

Ở **mục 2**, đổi `RUN_MODE = "debug"` thành `RUN_MODE = "full"` rồi **Run all**.

| | debug | full |
|---|---|---|
| bản ghi | ~400 (stratified theo nguồn×nhãn) | toàn bộ (ACS-ECG ~18k + PTB-XL ~22k) |
| epoch/model | 3 | 30 (early stopping patience 10) |
| batch size | 8 | 64 (GPU) / 16 (CPU) |
| số model train | 9 | 9 |

**Full run có thể mất vài giờ trên Colab free T4** (xem cảnh báo ở mục 13). Checkpoint-resume hoạt
động cho từng model riêng lẻ — nếu phiên Colab đứt giữa chừng, Run all lại: model nào đã có
checkpoint khớp cấu hình được bỏ qua, chỉ train tiếp model còn thiếu.

Cache tín hiệu ACS-ECG (`sig_full_...`) dùng chung với 01/03 nếu bản ghi + thứ tự trùng khớp —
không bắt buộc nhưng tiết kiệm đáng kể thời gian nếu bạn đã chạy một trong hai notebook đó trước.